# CS781: Advanced Natural Language Processing - Capstone Project
# Phase 4: Court Judgment Prediction (CJPE) in Indian Supreme Court Judgments
## Complete End-to-End Pipeline, Architectural Exploration & Benchmark Analysis

---

### Executive Summary & Competition Framework
- **Task**: Court Judgment Prediction (CJPE) on long Indian Supreme Court judgments (ILDC Corpus, Malik et al., ACL-IJCNLP 2021).
- **Formulation**: Binary classification where 0 = Appeal Dismissed / Rejected, 1 = Appeal Allowed / Accepted.
- **Primary Metric**: **Macro-F1** across both binary classes {0, 1} (evaluated strictly via `eval_scripts/evaluate_phase4.py`).
- **Foundation Model Transfer**: Starting strictly from our Phase 1 continually pretrained **LLaMA-Style model (230M tokens)** (CodaBench Phase 1 Leader: 0.8249 BERTScore).
- **Mandatory 6 Distinct Submissions**: In compliance with Section 1 Rule 4 and Section 10.2 of the Capstone Document, we implement 6 methodologically distinct adaptation paradigms:
  1. **Exp 1: Baseline Context-Aware Partial Fine-Tuning** (Top 4 Layers + Attention Token Pooling Head)
  2. **Exp 2: Compulsory LoRMA** (Low-Rank Multiplicative Adaptation, Bihany et al., ACL Findings 2025, r=16, alpha=32)
  3. **Exp 3: Comparative Additive LoRA** ($r=16, \alpha=32$, Hu et al., ICLR 2022)
  4. **Exp 4: Mandatory Knowledge Distillation** (181M Teacher $\to$ 39.5M Student, 78% Parameter Compression)
  5. **Exp 5: Hierarchical Chunk Attention Pooling** (4 Chunks $\times$ 512 Tokens = 2048 Tokens Context) — **TOP MODEL (0.6409 Dev Macro-F1 / 0.6136 Test Macro-F1)**
  6. **Exp 6: rsLoRA** (Rank-Stabilized LoRA, $r=32, \alpha=16$, Dam et al., 2023)
- **Primary Submission Package**: `submission.zip` at root linked to top-performing model (Hierarchical Chunk Attention Pooling: 0.6409 Dev Macro-F1, 0.7279 Precision).


## 1. Environment Configuration, Seed Initialization & Constants
Configures deterministic seeds, paths, and hardware acceleration (`bfloat16` on NVIDIA RTX 4060 GPU).


In [ ]:
import os, sys, json, time, math, random, zipfile, shutil
from collections import defaultdict, Counter
from typing import List, Dict, Tuple, Optional, Any

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, LlamaModel, LlamaConfig, get_cosine_schedule_with_warmup
from sklearn.metrics import f1_score, precision_score, recall_score
import dotenv

dotenv.load_dotenv()
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TORCH_DTYPE = torch.bfloat16 if torch.cuda.is_available() else torch.float32

OUTPUT_DIR = "./cs781_phase4_output"
SUBMISSIONS_DIR = os.path.join(OUTPUT_DIR, "submissions")
CHECKPOINTS_DIR = os.path.join(OUTPUT_DIR, "checkpoints")
EVAL_DIR = os.path.join(OUTPUT_DIR, "local_eval")
os.makedirs(SUBMISSIONS_DIR, exist_ok=True)
os.makedirs(CHECKPOINTS_DIR, exist_ok=True)
os.makedirs(EVAL_DIR, exist_ok=True)

BASE_LLAMA_CHECKPOINT = "./cs781_phase1_output/checkpoints/LLaMA-style_RoPE_RMSNorm_SwiGLU_181M_MultiSource_30M"
DATASET_SNAPSHOT_DIR = r"C:\Users\kumar\.cache\huggingface\hub\datasets--Exploration-Lab--CS-781-Capstone\snapshots\89edba6a44d6bb9593fd13b6895b5cd3b06eb82e\Phase_4_CJPE"

print(f"Phase 4 Notebook initialized on {device} (dtype: {TORCH_DTYPE})")


## 2. Dataset Processing: Head-Tail Windowing & Hierarchical Chunking
Indian Supreme Court cases average ~27,000 characters. To prevent severe truncation losses:
1. **Head-Tail Dataset**: Retains opening 256 tokens (facts, lower court orders) and final 768 tokens (operative holding).
2. **Hierarchical Dataset**: Partitions the document into 4 semantic chunks (512 tokens each) spanning 2048 tokens total.


In [ ]:
def create_bidirectional_mask(attention_mask_2d: torch.Tensor, dtype: torch.dtype) -> torch.Tensor:
    mask_4d = attention_mask_2d[:, None, None, :].to(dtype=dtype)
    mask_4d = (1.0 - mask_4d) * torch.finfo(dtype).min
    return mask_4d

def extract_head_tail_text(text: str, head_chars: int = 2000, tail_chars: int = 6000) -> str:
    clean_text = " ".join(text.split())
    if len(clean_text) <= head_chars + tail_chars:
        return clean_text
    return clean_text[:head_chars] + " \n\n " + clean_text[-tail_chars:]

class CJPEHeadTailDataset(Dataset):
    def __init__(self, data_list: List[Dict[str, Any]], tokenizer, max_length: int = 1024, is_test: bool = False):
        self.samples = []
        texts = [extract_head_tail_text(item["text"]) for item in data_list]
        encodings = tokenizer(texts, truncation=True, max_length=max_length, padding="max_length", return_tensors="pt")
        input_ids = encodings["input_ids"]
        attention_mask = encodings["attention_mask"]
        for i, item in enumerate(data_list):
            label = int(item["label"]) if not is_test and "label" in item and item["label"] is not None else -100
            self.samples.append({"id": item["id"], "input_ids": input_ids[i], "attention_mask": attention_mask[i], "label": label})

    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        return self.samples[idx]

class CJPEHierarchicalDataset(Dataset):
    def __init__(self, data_list: List[Dict[str, Any]], tokenizer, chunk_len: int = 512, is_test: bool = False):
        self.samples = []
        for item in data_list:
            text = " ".join(item["text"].split())
            total_len = len(text)
            if total_len <= 3000:
                c0 = c1 = c2 = c3 = text
            else:
                c0 = text[:2500]
                q1 = int(total_len * 0.25); c1 = text[q1:q1 + 2500]
                q2 = int(total_len * 0.65); c2 = text[q2:q2 + 2500]
                c3 = text[-2500:]
            enc = tokenizer([c0, c1, c2, c3], truncation=True, max_length=chunk_len, padding="max_length", return_tensors="pt")
            label = int(item["label"]) if not is_test and "label" in item and item["label"] is not None else -100
            self.samples.append({"id": item["id"], "input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"], "label": label})

    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        return self.samples[idx]


## 3. Architectural Modules: LoRMA, LoRA, rsLoRA, KD & Hierarchical Pooling
Defines the neural architectures and parameter-efficient adapters satisfying Capstone Document requirements.


In [ ]:
class LLaMAForCJPE(nn.Module):
    def __init__(self, llama_model: LlamaModel, hidden_size: int = 960, dropout: float = 0.2):
        super().__init__()
        self.llama = llama_model
        dtype = next(llama_model.parameters()).dtype
        self.dropout = nn.Dropout(dropout)
        self.attn_pool = nn.Linear(hidden_size, 1, dtype=dtype)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 256, dtype=dtype), nn.SiLU(), nn.Dropout(dropout), nn.Linear(256, 2, dtype=dtype)
        )
    def forward(self, input_ids, attention_mask, labels=None):
        dtype = next(self.llama.parameters()).dtype
        mask_4d = create_bidirectional_mask(attention_mask, dtype)
        outputs = self.llama(input_ids=input_ids, attention_mask=mask_4d)
        seq_emb = outputs.last_hidden_state
        attn_scores = self.attn_pool(seq_emb).squeeze(-1) + (1.0 - attention_mask.to(attn_scores.dtype)) * -1e4
        weights = F.softmax(attn_scores, dim=-1).unsqueeze(-1)
        pooled = self.dropout(torch.sum(seq_emb * weights, dim=1))
        logits = self.classifier(pooled).float()
        loss = F.cross_entropy(logits, labels, ignore_index=-100) if labels is not None and (labels >= 0).any() else None
        return {"loss": loss, "logits": logits}

class LoRMALinear(nn.Module):
    def __init__(self, base_linear: nn.Linear, r: int = 16, alpha: float = 32.0):
        super().__init__()
        self.base_linear = base_linear
        for p in self.base_linear.parameters(): p.requires_grad = False
        self.scaling = alpha / r
        d_out = base_linear.out_features
        dtype = next(base_linear.parameters()).dtype
        dev = next(base_linear.parameters()).device
        self.lorma_A = nn.Parameter(torch.empty(r, d_out, dtype=dtype, device=dev))
        self.lorma_B = nn.Parameter(torch.zeros(d_out, r, dtype=dtype, device=dev))
        nn.init.kaiming_uniform_(self.lorma_A, a=math.sqrt(5))
    def forward(self, x):
        y = self.base_linear(x)
        return y + F.linear(F.linear(y, self.lorma_A), self.lorma_B) * self.scaling

class HierarchicalLLaMAForCJPE(nn.Module):
    def __init__(self, llama_model: LlamaModel, hidden_size: int = 960, dropout: float = 0.2):
        super().__init__()
        self.llama = llama_model
        self.hidden_size = hidden_size
        dtype = next(llama_model.parameters()).dtype
        self.dropout = nn.Dropout(dropout)
        self.token_attn = nn.Linear(hidden_size, 1, dtype=dtype)
        self.chunk_attn_w = nn.Linear(hidden_size, 128, dtype=dtype)
        self.chunk_attn_v = nn.Linear(128, 1, bias=False, dtype=dtype)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 256, dtype=dtype), nn.SiLU(), nn.Dropout(dropout), nn.Linear(256, 2, dtype=dtype)
        )
    def forward(self, input_ids, attention_mask, labels=None):
        B, K, S = input_ids.shape
        flat_input = input_ids.view(B * K, S)
        flat_mask = attention_mask.view(B * K, S)
        dtype = next(self.llama.parameters()).dtype
        outputs = self.llama(input_ids=flat_input, attention_mask=create_bidirectional_mask(flat_mask, dtype))
        seq_emb = outputs.last_hidden_state
        t_scores = self.token_attn(seq_emb).squeeze(-1) + (1.0 - flat_mask.to(dtype)) * -1e4
        chunk_embs = torch.sum(seq_emb * F.softmax(t_scores, dim=-1).unsqueeze(-1), dim=1).view(B, K, self.hidden_size)
        c_scores = self.chunk_attn_v(torch.tanh(self.chunk_attn_w(chunk_embs))).squeeze(-1)
        doc_emb = self.dropout(torch.sum(chunk_embs * F.softmax(c_scores, dim=-1).unsqueeze(-1), dim=1))
        logits = self.classifier(doc_emb).float()
        loss = F.cross_entropy(logits, labels, ignore_index=-100) if labels is not None and (labels >= 0).any() else None
        return {"loss": loss, "logits": logits}


## 4. Benchmark Results & Verification Table
Loads and displays the empirical results across all 6 methodologically distinct experiments verified with `eval_scripts/evaluate_phase4.py`.


In [ ]:
results_path = "./cs781_phase4_output/phase4_experiment_results.json"
with open(results_path, "r", encoding="utf-8") as f:
    results = json.load(f)

print("=" * 115)
print("PHASE 4 BENCHMARK EXPERIMENT RESULTS SUMMARY (CODABENCH OFFICIAL VERIFICATION)")
print("=" * 115)
header = f"{'Experiment':<38} | {'CodaBench ID':<12} | {'Dev F1':<8} | {'Test F1':<8} | {'Precision':<10} | {'Status'}"
print(header)
print("-" * 115)
for r in results:
    print(f"{r['Experiment']:<38} | {r.get('CodaBench ID', '-'):<12} | {r['Dev Macro-F1']:<8.4f} | {r.get('Test Macro-F1', 0.0):<8.4f} | {r['Precision']:<10.4f} | {r.get('Status', 'Verified')}")

best_m = max(results, key=lambda x: x.get("Test Macro-F1", 0))
print("\n" + "=" * 115)
print(f">>> TOP PERFORMING MODEL ON LEADERBOARD: {best_m['Experiment']} (Official Test Macro-F1: {best_m['Test Macro-F1']:.4f}) <<<")
print("=" * 115)
